# Lesson 3: BERT-based NER with Hugging Face Transformers

## 🎯 Learning Objectives

By the end of this lesson, you will:
1. Understand how BERT works for token classification/NER
2. Use pre-trained BERT NER models with Hugging Face
3. Master the token classification pipeline
4. Handle subword tokenization challenges
5. Understand attention mechanisms in NER context

---

## 📚 Table of Contents

1. [Introduction to BERT](#1-introduction-to-bert)
2. [BERT for Token Classification](#2-bert-for-token-classification)
3. [Using Pre-trained NER Models](#3-using-pre-trained-ner-models)
4. [Understanding Subword Tokenization](#4-understanding-subword-tokenization)
5. [The NER Pipeline in Detail](#5-the-ner-pipeline-in-detail)
6. [Exploring Model Internals](#6-exploring-model-internals)
7. [Comparing BERT Variants](#7-comparing-bert-variants)
8. [Further Reading](#8-further-reading)

---

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q transformers torch datasets seqeval accelerate

In [ ]:
# Import libraries
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline,
    BertTokenizer,
    BertForTokenClassification,
    BertModel,
    BertConfig
)
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")
print(f"✅ PyTorch version: {torch.__version__}")

---

## 1. Introduction to BERT

### What is BERT?

> **BERT** (Bidirectional Encoder Representations from Transformers) is a transformer-based language model developed by Google. It uses bidirectional training to understand context from both left and right directions simultaneously.
>
> — [Devlin et al., 2019](https://arxiv.org/abs/1810.04805)

### Key Innovations

| Feature | Description |
|---------|-------------|
| **Bidirectional** | Reads text in both directions simultaneously |
| **Pre-training** | Trained on massive text corpora (Wikipedia, BookCorpus) |
| **Transfer Learning** | Fine-tune for downstream tasks |
| **Attention Mechanism** | Captures long-range dependencies |

### BERT Architecture

```
                              ┌─────────────────┐
                              │  Output Layer   │
                              │  (Task Head)    │
                              └────────┬────────┘
                                       │
                              ┌────────▼────────┐
                              │  Transformer    │
                              │  Encoder x12    │
                              │  (or x24)       │
                              └────────┬────────┘
                                       │
          ┌────────────────────────────┼────────────────────────────┐
          │                            │                            │
┌─────────▼─────────┐      ┌──────────▼──────────┐      ┌─────────▼─────────┐
│ Token Embeddings  │  +   │ Position Embeddings │  +   │ Segment Embeddings│
└─────────┬─────────┘      └──────────┬──────────┘      └─────────┬─────────┘
          │                            │                            │
          └────────────────────────────┴────────────────────────────┘
                                       │
                              ┌────────▼────────┐
                              │  [CLS] tok1 tok2│
                              │  ... tokn [SEP] │
                              └─────────────────┘
                                   Input Text
```

### BERT Model Sizes

| Model | Layers | Hidden | Heads | Parameters |
|-------|--------|--------|-------|------------|
| BERT-Base | 12 | 768 | 12 | 110M |
| BERT-Large | 24 | 1024 | 16 | 340M |

In [ ]:
# Explore BERT configuration
config = BertConfig.from_pretrained("bert-base-cased")

print("🔧 BERT-Base Configuration:\n")
print(f"   Hidden size: {config.hidden_size}")
print(f"   Number of layers: {config.num_hidden_layers}")
print(f"   Attention heads: {config.num_attention_heads}")
print(f"   Intermediate size: {config.intermediate_size}")
print(f"   Vocabulary size: {config.vocab_size}")
print(f"   Max position embeddings: {config.max_position_embeddings}")

---

## 2. BERT for Token Classification

### How BERT is Adapted for NER

For NER (token classification), we add a classification head on top of BERT:

```
Input: [CLS] Barack Obama visited Paris [SEP]
           ↓
       BERT Encoder (12 layers)
           ↓
Token representations: [h_cls, h_barack, h_obama, h_visited, h_paris, h_sep]
           ↓
       Linear Layer (768 → num_labels)
           ↓
Output:    [_,    B-PER,   I-PER,    O,      B-LOC,    _]
```

### Mathematical Formulation

For each token $i$:

$$P(y_i | x) = \text{softmax}(W \cdot h_i + b)$$

Where:
- $h_i$ is the hidden representation from BERT's last layer
- $W$ is the classification weight matrix
- $b$ is the bias vector

In [ ]:
# Load a BERT model for token classification
model_name = "dslim/bert-base-NER"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

print(f"✅ Loaded model: {model_name}")
print(f"\n📊 Model Architecture:")
print(f"   Base model: {model.config.model_type}")
print(f"   Number of labels: {model.config.num_labels}")
print(f"   Label mapping: {model.config.id2label}")

In [ ]:
# Manual forward pass to understand the process
text = "Barack Obama visited Paris last summer."

# Step 1: Tokenize
inputs = tokenizer(text, return_tensors="pt")

print("Step 1: Tokenization")
print(f"   Input IDs: {inputs['input_ids']}")
print(f"   Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
print(f"   Attention mask: {inputs['attention_mask']}")

# Step 2: Forward pass
with torch.no_grad():
    outputs = model(**inputs)

print(f"\nStep 2: Model Output")
print(f"   Logits shape: {outputs.logits.shape}")
print(f"   (batch_size, sequence_length, num_labels)")

# Step 3: Get predictions
predictions = torch.argmax(outputs.logits, dim=2)

print(f"\nStep 3: Predictions")
print(f"   Predicted label IDs: {predictions[0].tolist()}")

# Step 4: Convert to labels
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
labels = [model.config.id2label[p.item()] for p in predictions[0]]

print(f"\nStep 4: Final Output")
print(f"{'Token':<15} {'Label':<10}")
print("-" * 25)
for token, label in zip(tokens, labels):
    print(f"{token:<15} {label:<10}")

---

## 3. Using Pre-trained NER Models

### The Easy Way: Pipeline API

Hugging Face provides a simple `pipeline` API for common tasks:

In [ ]:
# Create NER pipeline
ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"  # Group subwords into entities
)

# Test on sample text
text = "Apple CEO Tim Cook announced new products in Cupertino, California."

results = ner_pipeline(text)

print("🏷️ NER Pipeline Results:\n")
print(f"Text: {text}\n")

for entity in results:
    print(f"   Entity: '{entity['word']}'")
    print(f"   Label: {entity['entity_group']}")
    print(f"   Score: {entity['score']:.4f}")
    print(f"   Position: [{entity['start']}, {entity['end']}]")
    print()

In [ ]:
# Different aggregation strategies
strategies = ["none", "simple", "first", "average", "max"]

text = "Elon Musk founded SpaceX and Tesla."

print("📊 Aggregation Strategy Comparison:\n")
print(f"Text: {text}\n")

for strategy in strategies:
    pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy=strategy)
    results = pipe(text)
    
    print(f"Strategy: {strategy}")
    if strategy == "none":
        # No aggregation - shows individual tokens
        entities = [(r['word'], r['entity']) for r in results[:5]]
    else:
        entities = [(r['word'], r['entity_group']) for r in results]
    print(f"   Entities: {entities}")
    print()

In [ ]:
# Batch processing with pipeline
texts = [
    "Google was founded by Larry Page and Sergey Brin.",
    "Microsoft CEO Satya Nadella spoke at the conference in Seattle.",
    "Amazon's headquarters are in Seattle, Washington.",
    "Mark Zuckerberg founded Facebook in his Harvard dorm room."
]

print("⚡ Batch Processing:\n")

# Process batch
batch_results = ner_pipeline(texts)

for text, entities in zip(texts, batch_results):
    print(f"Text: {text}")
    entity_list = [(e['word'], e['entity_group']) for e in entities]
    print(f"Entities: {entity_list}\n")

### Popular Pre-trained NER Models

| Model | Description | Labels |
|-------|-------------|--------|
| `dslim/bert-base-NER` | BERT fine-tuned on CoNLL-2003 | PER, LOC, ORG, MISC |
| `dslim/bert-large-NER` | Larger BERT for better accuracy | PER, LOC, ORG, MISC |
| `dbmdz/bert-large-cased-finetuned-conll03-english` | German team's BERT | PER, LOC, ORG, MISC |
| `Jean-Baptiste/roberta-large-ner-english` | RoBERTa-based NER | PER, LOC, ORG, MISC |
| `flair/ner-english-large` | Flair's NER model | PER, LOC, ORG, MISC |

> **Reference**: [Hugging Face Model Hub - NER](https://huggingface.co/models?pipeline_tag=token-classification)

In [ ]:
# Compare different models
models_to_compare = [
    "dslim/bert-base-NER",
    "dslim/bert-large-NER"
]

test_text = "The European Union's headquarters in Brussels hosted a meeting between Angela Merkel and Emmanuel Macron."

print("🔍 Model Comparison:\n")
print(f"Text: {test_text}\n")

for model_name in models_to_compare:
    pipe = pipeline("ner", model=model_name, aggregation_strategy="simple")
    results = pipe(test_text)
    
    print(f"Model: {model_name}")
    for entity in results:
        print(f"   • {entity['word']:<25} → {entity['entity_group']} ({entity['score']:.3f})")
    print()

---

## 4. Understanding Subword Tokenization

### The Challenge

BERT uses **WordPiece tokenization**, which breaks words into subwords:

```
"playing" → ["play", "##ing"]
"unhappiness" → ["un", "##happiness"] or ["un", "##happy", "##ness"]
```

This creates a mismatch between:
- **Word-level labels** (what we want): `["O", "B-PER", "I-PER", "O"]`
- **Subword tokens** (what BERT sees): More tokens due to splitting

### Solutions

1. **First subword only**: Only the first subword gets the label
2. **All subwords**: All subwords get the same label
3. **Aggregation**: Combine subword predictions back to word-level

In [ ]:
# Demonstrate subword tokenization
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

words_to_tokenize = [
    "hello",
    "playing",
    "unhappiness",
    "Constantinople",
    "Schwarzenegger",
    "COVID-19"
]

print("🔤 Subword Tokenization Examples:\n")
print(f"{'Word':<20} {'Tokens':<40} {'# Subwords'}")
print("=" * 70)

for word in words_to_tokenize:
    tokens = tokenizer.tokenize(word)
    print(f"{word:<20} {str(tokens):<40} {len(tokens)}")

In [ ]:
# The alignment challenge
text = "Arnold Schwarzenegger visited Vienna."

# Original words
original_words = text.split()
word_labels = ["B-PER", "I-PER", "O", "B-LOC", "O"]

# Tokenize
tokens = tokenizer.tokenize(text)

print("🔗 The Alignment Challenge:\n")
print("Original words and labels:")
for word, label in zip(original_words, word_labels):
    print(f"   {word:<20} → {label}")

print(f"\nSubword tokens: {tokens}")
print(f"\nProblem: {len(original_words)} words → {len(tokens)} tokens")
print("We need to align labels with subword tokens!")

In [ ]:
# Solution: Use word_ids for alignment
def tokenize_and_align(text, word_labels, tokenizer):
    """Tokenize text and align labels with subwords"""
    
    # Split into words first
    words = text.split()
    
    # Tokenize with word tracking
    tokenized = tokenizer(
        words,
        is_split_into_words=True,  # Important!
        return_tensors="pt"
    )
    
    # Get word IDs (which word each token belongs to)
    word_ids = tokenized.word_ids()
    
    # Align labels
    aligned_labels = []
    previous_word_idx = None
    
    for word_idx in word_ids:
        if word_idx is None:
            # Special tokens ([CLS], [SEP])
            aligned_labels.append(-100)  # Ignored in loss
        elif word_idx != previous_word_idx:
            # First subword of a new word
            aligned_labels.append(word_labels[word_idx] if word_idx < len(word_labels) else "O")
        else:
            # Continuation subword (same word)
            aligned_labels.append(-100)  # Or use I- prefix
        
        previous_word_idx = word_idx
    
    return tokenized, word_ids, aligned_labels

# Test the alignment
text = "Arnold Schwarzenegger visited Vienna."
word_labels = ["B-PER", "I-PER", "O", "B-LOC", "O"]

tokenized, word_ids, aligned_labels = tokenize_and_align(text, word_labels, tokenizer)

tokens = tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])

print("✅ Aligned Tokenization:\n")
print(f"{'Token':<20} {'Word ID':<10} {'Label'}")
print("=" * 45)
for token, wid, label in zip(tokens, word_ids, aligned_labels):
    label_str = str(label) if label != -100 else "[IGNORE]"
    wid_str = str(wid) if wid is not None else "None"
    print(f"{token:<20} {wid_str:<10} {label_str}")

---

## 5. The NER Pipeline in Detail

Let's build our own NER prediction function to understand every step:

In [ ]:
class BERTNERPredictor:
    """A detailed BERT NER predictor for educational purposes"""
    
    def __init__(self, model_name="dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.eval()
        
        # Get label mappings
        self.id2label = self.model.config.id2label
        self.label2id = self.model.config.label2id
    
    def predict_detailed(self, text):
        """Predict with detailed step-by-step output"""
        
        print("=" * 70)
        print("BERT NER PREDICTION PIPELINE")
        print("=" * 70)
        
        # Step 1: Tokenization
        print("\n📝 STEP 1: Tokenization")
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            return_offsets_mapping=True
        )
        
        tokens = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        offsets = inputs['offset_mapping'][0].tolist()
        
        print(f"   Original text: '{text}'")
        print(f"   Tokens ({len(tokens)}): {tokens}")
        
        # Step 2: Model forward pass
        print("\n🧠 STEP 2: Model Forward Pass")
        
        # Remove offset_mapping before passing to model
        model_inputs = {k: v for k, v in inputs.items() if k != 'offset_mapping'}
        
        with torch.no_grad():
            outputs = self.model(**model_inputs)
        
        logits = outputs.logits[0]  # Shape: (seq_len, num_labels)
        print(f"   Logits shape: {logits.shape}")
        print(f"   Number of classes: {logits.shape[1]}")
        
        # Step 3: Softmax probabilities
        print("\n📊 STEP 3: Convert to Probabilities")
        probs = torch.softmax(logits, dim=-1)
        
        # Step 4: Get predictions
        print("\n🎯 STEP 4: Get Predictions")
        predictions = torch.argmax(probs, dim=-1)
        confidence = torch.max(probs, dim=-1).values
        
        # Step 5: Convert to labels
        print("\n🏷️ STEP 5: Token-level Results")
        print(f"{'Token':<15} {'Prediction':<10} {'Confidence':<12} {'Offset'}")
        print("-" * 55)
        
        results = []
        for i, (token, pred, conf, offset) in enumerate(zip(tokens, predictions, confidence, offsets)):
            label = self.id2label[pred.item()]
            print(f"{token:<15} {label:<10} {conf.item():.4f}       {offset}")
            
            if label != 'O' and token not in ['[CLS]', '[SEP]']:
                results.append({
                    'token': token,
                    'label': label,
                    'confidence': conf.item(),
                    'start': offset[0],
                    'end': offset[1]
                })
        
        # Step 6: Aggregate entities
        print("\n🔗 STEP 6: Aggregated Entities")
        entities = self._aggregate_entities(results, text)
        
        for ent in entities:
            print(f"   • '{ent['text']}' → {ent['label']} (confidence: {ent['confidence']:.4f})")
        
        return entities
    
    def _aggregate_entities(self, token_results, original_text):
        """Aggregate subword tokens into entities"""
        if not token_results:
            return []
        
        entities = []
        current_entity = None
        
        for result in token_results:
            label = result['label']
            
            if label.startswith('B-'):
                # Save previous entity
                if current_entity:
                    current_entity['text'] = original_text[current_entity['start']:current_entity['end']]
                    entities.append(current_entity)
                
                # Start new entity
                current_entity = {
                    'label': label[2:],
                    'start': result['start'],
                    'end': result['end'],
                    'confidence': result['confidence']
                }
            
            elif label.startswith('I-') and current_entity:
                # Continue current entity
                current_entity['end'] = result['end']
                current_entity['confidence'] = min(current_entity['confidence'], result['confidence'])
        
        # Don't forget last entity
        if current_entity:
            current_entity['text'] = original_text[current_entity['start']:current_entity['end']]
            entities.append(current_entity)
        
        return entities

# Create predictor and test
predictor = BERTNERPredictor()

text = "Barack Obama visited the Eiffel Tower in Paris."
entities = predictor.predict_detailed(text)

---

## 6. Exploring Model Internals

### Attention Visualization

Let's visualize what BERT "pays attention to" when making NER predictions:

In [ ]:
import matplotlib.pyplot as plt

# Load model with attention outputs
model_name = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    output_attentions=True
)
model.eval()

text = "Steve Jobs founded Apple in California."

# Get attention weights
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# outputs.attentions is a tuple of attention weights for each layer
# Shape: (batch, heads, seq_len, seq_len)
attentions = outputs.attentions

print(f"Number of layers: {len(attentions)}")
print(f"Attention shape per layer: {attentions[0].shape}")

tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

In [ ]:
# Visualize attention from last layer
def plot_attention(attention, tokens, layer_idx=11, head_idx=0):
    """Plot attention weights"""
    
    # Get specific layer and head
    attn = attention[layer_idx][0, head_idx].numpy()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    im = ax.imshow(attn, cmap='Blues')
    
    # Set ticks
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)
    
    ax.set_xlabel('Key (attending to)')
    ax.set_ylabel('Query (from)')
    ax.set_title(f'Attention Weights - Layer {layer_idx + 1}, Head {head_idx + 1}')
    
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()

# Plot attention from last layer, first head
plot_attention(attentions, tokens, layer_idx=11, head_idx=0)

In [ ]:
# Average attention across all heads in last layer
def plot_average_attention(attention, tokens, layer_idx=11):
    """Plot average attention across all heads"""
    
    # Average across heads
    attn = attention[layer_idx][0].mean(dim=0).numpy()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    im = ax.imshow(attn, cmap='Reds')
    
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)
    
    ax.set_xlabel('Key (attending to)')
    ax.set_ylabel('Query (from)')
    ax.set_title(f'Average Attention - Layer {layer_idx + 1}')
    
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()
    
    # Print key observations
    print("\n🔍 Key Observations:")
    print("   • High attention on diagonal = self-attention")
    print("   • 'Steve' and 'Jobs' likely attend to each other (same entity)")
    print("   • Entity tokens may attend to context words")

plot_average_attention(attentions, tokens)

In [ ]:
# Analyze hidden states
model_hidden = AutoModelForTokenClassification.from_pretrained(
    model_name,
    output_hidden_states=True
)
model_hidden.eval()

with torch.no_grad():
    outputs_hidden = model_hidden(**inputs)

hidden_states = outputs_hidden.hidden_states

print(f"Number of hidden state layers: {len(hidden_states)}")
print(f"Hidden state shape: {hidden_states[0].shape}")
print(f"(batch_size, sequence_length, hidden_size)")

# Compare first and last layer representations
first_layer = hidden_states[0][0]  # After embedding
last_layer = hidden_states[-1][0]  # Before classification

# Compute similarity between token representations
from torch.nn.functional import cosine_similarity

print("\n📊 Token Representation Similarity (Last Layer):")
print("   Comparing 'Steve' (idx 1) with other tokens:\n")

steve_repr = last_layer[1]  # 'Steve' token

for i, token in enumerate(tokens):
    sim = cosine_similarity(steve_repr.unsqueeze(0), last_layer[i].unsqueeze(0))
    print(f"   'Steve' <-> '{token}': {sim.item():.4f}")

---

## 7. Comparing BERT Variants

### Popular Transformer Models for NER

| Model | Year | Key Innovation | NER Performance |
|-------|------|---------------|----------------|
| BERT | 2018 | Bidirectional pre-training | Baseline |
| RoBERTa | 2019 | More data, no NSP | +0.3-0.5% F1 |
| ALBERT | 2019 | Parameter sharing | Similar, smaller |
| DistilBERT | 2019 | Knowledge distillation | 97% perf, 60% params |
| ELECTRA | 2020 | Replaced token detection | +0.5% F1 |
| DeBERTa | 2021 | Disentangled attention | +1% F1 |

In [ ]:
# Compare model sizes and inference speed
import time

models_info = [
    ("dslim/bert-base-NER", "BERT-base"),
    ("dslim/distilbert-NER", "DistilBERT"),
]

test_text = "The European Central Bank president Christine Lagarde spoke in Frankfurt."
num_runs = 10

print("⚡ Model Comparison:\n")
print(f"{'Model':<20} {'Parameters':<15} {'Avg Time (ms)':<15} {'Entities'}")
print("=" * 80)

for model_path, model_name in models_info:
    try:
        # Load model
        pipe = pipeline("ner", model=model_path, aggregation_strategy="simple")
        
        # Count parameters
        num_params = sum(p.numel() for p in pipe.model.parameters())
        params_str = f"{num_params / 1e6:.1f}M"
        
        # Measure inference time
        times = []
        for _ in range(num_runs):
            start = time.time()
            results = pipe(test_text)
            times.append((time.time() - start) * 1000)
        
        avg_time = sum(times) / len(times)
        entities = [(e['word'], e['entity_group']) for e in results]
        
        print(f"{model_name:<20} {params_str:<15} {avg_time:.2f}ms{' '*7} {entities}")
        
    except Exception as e:
        print(f"{model_name:<20} Error: {str(e)[:50]}")

---

## 8. Further Reading

### 📚 Essential Papers

1. **BERT: Pre-training of Deep Bidirectional Transformers** (2019)
   - Devlin, J., Chang, M. W., Lee, K., & Toutanova, K.
   - [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)

2. **Attention Is All You Need** (2017)
   - Vaswani, A., et al.
   - [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)

3. **RoBERTa: A Robustly Optimized BERT Pretraining Approach** (2019)
   - Liu, Y., et al.
   - [arXiv:1907.11692](https://arxiv.org/abs/1907.11692)

4. **DeBERTa: Decoding-enhanced BERT with Disentangled Attention** (2021)
   - He, P., et al.
   - [arXiv:2006.03654](https://arxiv.org/abs/2006.03654)

### 🔗 Official Documentation

- [Hugging Face Transformers](https://huggingface.co/docs/transformers/)
- [Token Classification Guide](https://huggingface.co/docs/transformers/tasks/token_classification)
- [BERT Documentation](https://huggingface.co/docs/transformers/model_doc/bert)

### 🎓 Tutorials

- [Hugging Face NLP Course - Chapter 7](https://huggingface.co/learn/llm-course/en/chapter7/2)
- [The Illustrated BERT](http://jalammar.github.io/illustrated-bert/)
- [The Annotated Transformer](http://nlp.seas.harvard.edu/2018/04/03/attention.html)

---

## ✅ Lesson Summary

In this lesson, we covered:

1. **BERT Architecture**: How BERT works and its key innovations
2. **Token Classification**: Adapting BERT for NER with a classification head
3. **Pre-trained Models**: Using the pipeline API and comparing models
4. **Subword Tokenization**: Handling the alignment challenge
5. **Model Internals**: Visualizing attention and hidden states
6. **Model Comparison**: Trade-offs between different transformer variants

### 🚀 Next Lesson Preview

In **Lesson 4**, we'll explore **GLiNER - Zero-Shot NER**, including:
- How GLiNER enables NER without training
- Using custom entity types on the fly
- Comparing zero-shot vs fine-tuned approaches

In [ ]:
print("🎉 Congratulations! You've completed Lesson 3: BERT-based NER")
print("\n📝 Key takeaways:")
print("   1. BERT uses bidirectional attention for context understanding")
print("   2. NER is token classification with a linear head on top of BERT")
print("   3. Subword tokenization requires careful label alignment")
print("   4. The pipeline API makes inference easy")
print("   5. Different BERT variants offer speed/accuracy trade-offs")
print("\n👉 Continue to Lesson 4: GLiNER - Zero-Shot NER")